In [ ]:
import os
import requests
import re
import json
import time
from datetime import datetime

with open(".env", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("GITHUB_TOKEN="):
            TOKEN = line.strip().split("=", 1)[1].strip().strip("'").strip('"')
            break

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json"
}

def parse_pr_link(link):
    pattern = r"https://github\.com/([^/]+)/([^/]+)/pull/(\d+)"
    match = re.match(pattern, link.strip())
    if match:
        owner, repo, pr_number = match.groups()
        return owner, repo, int(pr_number)
    return None, None, None

def handle_rate_limit(response):
    if response.status_code == 403 and "X-RateLimit-Remaining" in response.headers:
        remaining = int(response.headers.get("X-RateLimit-Remaining", 0))
        reset_ts = int(response.headers.get("X-RateLimit-Reset", 0))
        now_ts = int(time.time())

        if remaining == 0:
            wait_time = reset_ts - now_ts + 5
            reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')
            print(f"API limit. Waiting {wait_time}s (reset on {reset_time} UTC)...")
            time.sleep(max(wait_time, 0))
            return True
    return False

with open("pr_links.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

output_path = "pr_lines_summary.txt"
batch_size = 5000
total_lines = len(lines)

print(f"Found {total_lines} link PRs. Batching {batch_size} per part...")

with open(output_path, "a", encoding="utf-8") as out:
    for i in range(0, total_lines, batch_size):
        batch = lines[i:i + batch_size]
        print(f"\Processing part {i // batch_size + 1} ({len(batch)} links)...")

        for line in batch:
            parts = line.split()
            if len(parts) != 2:
                print(f"Wrong format: {line}")
                continue

            pr_id, link = parts
            owner, repo, pr_number = parse_pr_link(link)
            if not owner:
                print(f"Wrong link: {link}")
                continue

            url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"

            # Obsluga limitow
            while True:
                response = requests.get(url, headers=headers)
                if handle_rate_limit(response):
                    continue

                if response.status_code != 200:
                    print(f"Error {response.status_code} while fetching: {url}")
                    break

                pr_data = response.json()
                additions = pr_data.get("additions", 0)
                deletions = pr_data.get("deletions", 0)
                changed_lines = additions + deletions

                record = {
                    "pr_id": pr_id,
                    "pr_link": link,
                    "additions": additions,
                    "deletions": deletions,
                    "changed_lines": changed_lines
                }

                out.write(json.dumps(record, ensure_ascii=False) + "\n")
                break  # wyjscie z petli while True

        if i + batch_size < total_lines:
            print(f"Quick break 10s before part ({i + batch_size}/{total_lines})...")
            time.sleep(10)

print(f"\nCompleted. Data save to: {output_path}")

Znaleziono 6618 linków PR. Przetwarzanie po 5000 na partię...

Przetwarzanie partii 1 (5000 linków)...


/tmp/ipykernel_4004069/4214724644.py:35: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')


Limit API osiągnięty. Czekam 667 sek. (reset o 2025-10-23 08:22:35 UTC)...
Krótka przerwa 10 sekund przed kolejną partią (5000/6618)...

Przetwarzanie partii 2 (1618 linków)...

Zakończono. Dane zapisano do pliku: pr_lines_summary.txt


In [ ]:
# Covert to parquet
import pandas as pd
import json

input_file = "pr_lines_summary.txt"
output_file = "human_pr_lines_summary.parquet"

records = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
df.to_parquet(output_file, index=False)

print(f"Writing data to parquet file: {output_file}")
print(f"Data count: {len(df)}")

Zapisano dane do pliku Parquet: human_pr_lines_summary.parquet
Liczba rekordów: 6618
